# Diffusion Strings with BioEmu

This notebook uses a pretrained BioEmu score model as the vector field for the dynamic SE(3) string method. The example uses Chignolin (`GYDPETGTWG`), but the `sequence` parameter can be changed to another protein monomer.

The workflow is:

1. Prepare BioEmu sequence embeddings.
2. Load a pretrained BioEmu checkpoint and its SDEs.
3. Wrap the model as paired translation/rotation `b_field` and `score_field` callbacks.
4. Generate ordinary BioEmu samples to initialize an SE(3) string.
5. Refine the string with the finite-temperature dynamic string method.
6. Save NumPy arrays plus BioEmu-compatible PDB/XTC trajectories.

Run this notebook from the `Diffusion-Strings` conda environment. A CUDA GPU is strongly recommended. On first use, BioEmu downloads its checkpoint and embedding-model assets, so embedding generation needs network access and several GB of free disk space.

## 1. Paths and configuration

The defaults are deliberately small enough for a first run. Increase the sample, string, walker, and timestep counts for an actual experiment.

In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "diffusion_strings").is_dir():
    raise RuntimeError("Start Jupyter in Diffusion-Strings or its notebooks directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

output_dir = repo_root / "outputs" / "bioemu_chignolin_dynamic_se3"
cache_embeds_dir = output_dir / "embeds_cache"
cache_so3_dir = output_dir / "so3_cache"
for directory in (output_dir, cache_embeds_dir, cache_so3_dir):
    directory.mkdir(parents=True, exist_ok=True)

# Matplotlib's normal user cache may be read-only on a compute node.
os.environ.setdefault("MPLCONFIGDIR", str(output_dir / "matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

sequence = "GYDPETGTWG"  # Chignolin
model_name = "bioemu-v1.1"
base_seed = 1234

# Set these to existing .npy files to avoid generating embeddings.
single_embeds_source = None
pair_embeds_source = None
allow_embedding_generation = True
msa_file = None  # Optional local .a3m file.

# Small demonstration settings. Production runs should use more steps/samples.
n_initial_samples = 8
n_string_points = 6
n_denoiser_steps = 50
n_walkers = 4
n_timesteps_total = 8
convergence_time_values = [0.35, 0.70]
T = 0.02
n_voronoi_cycles = 2
n_steps_per_cycle = 2
main_step_size = 0.01
update_coefficient = 0.25
corr_coeff = 1.0

print(f"repository: {repo_root}")
print(f"outputs:    {output_dir}")

## 2. Imports and device

`bioemu_vector_fields_from_model` handles BioEmu's time reversal, score scaling, probability-flow drift, and batching. The returned fields operate jointly on C-alpha translations and residue-frame rotations.

In [ ]:
import hydra
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

from bioemu.convert_chemgraph import save_pdb_and_xtc
from bioemu.model_utils import load_model, load_sdes, maybe_download_checkpoint
from bioemu.sample import DEFAULT_DENOISER_CONFIG_DIR, generate_batch

from diffusion_strings.adapters.from_bioemu import (
    bioemu_vector_fields_from_model,
    prepare_bioemu_embedding_cache,
)
from diffusion_strings.dynamics_se3 import dynamic_string_method_Tfinite_singular
from diffusion_strings.reparametrization import uniform_string_repametrize_se3

torch.set_grad_enabled(False)
torch.manual_seed(base_seed)
np.random.seed(base_seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
convergence_times = torch.tensor(
    convergence_time_values, device=device, dtype=dtype
)
convergence_weights = torch.ones_like(convergence_times)

print(f"PyTorch: {torch.__version__}")
print(f"device:  {device}")

## 3. Prepare BioEmu embeddings

BioEmu needs single-residue and pair embeddings for the sequence. This cell reuses the hashed cache when available. Otherwise it can either import the two files configured above or ask BioEmu to generate them.

If automatic generation is unsuitable on your machine, set `allow_embedding_generation = False` and provide both `single_embeds_source` and `pair_embeds_source`.

In [ ]:
single_embeds_file, pair_embeds_file = prepare_bioemu_embedding_cache(
    sequence=sequence,
    cache_embeds_dir=cache_embeds_dir,
    single_embeds_file=single_embeds_source,
    pair_embeds_file=pair_embeds_source,
    msa_file=msa_file,
    allow_embedding_generation=allow_embedding_generation,
)

print(f"single embeddings: {single_embeds_file}")
print(f"pair embeddings:   {pair_embeds_file}")

## 4. Load BioEmu

In [ ]:
ckpt_path, model_config_path = maybe_download_checkpoint(
    model_name=model_name,
    ckpt_path=None,
    model_config_path=None,
)

score_model = load_model(ckpt_path, model_config_path).to(device)
score_model.eval()

sdes = load_sdes(
    model_config_path=model_config_path,
    cache_so3_dir=cache_so3_dir,
)
for sde in sdes.values():
    if isinstance(sde, torch.nn.Module):
        sde.to(device)

print(f"checkpoint: {ckpt_path}")
print(f"config:     {model_config_path}")

## 5. Build diffusion-string fields

The adapter returns two callbacks with the signature

```python
translation_field, rotation_field = field(pos, node_orientations, t)
```

where the rotation component is an SO(3) tangent vector, not another rotation matrix.

In [ ]:
b_field, score_field = bioemu_vector_fields_from_model(
    score_model=score_model,
    sequence=sequence,
    sdes=sdes,
    cache_embeds_dir=cache_embeds_dir,
    single_embeds_file=single_embeds_file,
    pair_embeds_file=pair_embeds_file,
    cache_so3_dir=cache_so3_dir,
    msa_file=msa_file,
    device=device,
)

print("BioEmu b_field and score_field are ready.")

## 6. Seed an initial string with BioEmu samples

We first draw ordinary BioEmu samples, then redistribute them uniformly along coupled translation/rotation arc length. This gives the string method a valid sequence of SE(3) states.

In [ ]:
denoiser_config_path = DEFAULT_DENOISER_CONFIG_DIR / "dpm.yaml"
with denoiser_config_path.open() as handle:
    denoiser_config = yaml.safe_load(handle)
denoiser_config["N"] = n_denoiser_steps
denoiser = hydra.utils.instantiate(denoiser_config)

initial_batch = generate_batch(
    score_model=score_model,
    sequence=sequence,
    sdes=sdes,
    batch_size=n_initial_samples,
    seed=base_seed,
    denoiser=denoiser,
    cache_embeds_dir=cache_embeds_dir,
    msa_file=msa_file,
)

initial_t = initial_batch["pos"].to(device=device, dtype=dtype)
initial_r = initial_batch["node_orientations"].to(device=device, dtype=dtype)

string_t, string_r = uniform_string_repametrize_se3(
    initial_t,
    initial_r,
    n_new=n_string_points,
    corr_coeff=corr_coeff,
)

print("BioEmu samples:", tuple(initial_t.shape), tuple(initial_r.shape))
print("initial string:", tuple(string_t.shape), tuple(string_r.shape))

## 7. Smoke-test the adapter

This evaluates each field once before starting the longer string calculation. All returned components should have shape `(batch, residues, 3)` and contain finite values.

In [ ]:
test_t = torch.tensor(0.5, device=device, dtype=dtype)
test_pos = string_t[:2]
test_rot = string_r[:2]

b_pos, b_rot = b_field(test_pos, test_rot, test_t)
s_pos, s_rot = score_field(test_pos, test_rot, test_t)

for name, value in {
    "b_pos": b_pos,
    "b_rot": b_rot,
    "score_pos": s_pos,
    "score_rot": s_rot,
}.items():
    assert value.shape == test_pos.shape, (name, value.shape)
    assert torch.isfinite(value).all(), f"{name} contains non-finite values"
    print(f"{name:>9}: shape={tuple(value.shape)}, rms={value.square().mean().sqrt():.4g}")

## 8. Run the finite-temperature dynamic string method

The method transports the principal string with BioEmu's probability-flow field and applies score-driven finite-temperature walker updates at the configured convergence times.

In [ ]:
final_string_t, final_string_r, walker_t, walker_r = (
    dynamic_string_method_Tfinite_singular(
        string_t,
        string_r,
        b_field,
        score_field,
        T=T,
        n_timesteps_total=n_timesteps_total,
        convergence_times=convergence_times,
        convergence_weights=convergence_weights,
        n_walkers=n_walkers,
        n_voronoi_cycles=n_voronoi_cycles,
        n_steps_per_cycle=n_steps_per_cycle,
        main_step_size=main_step_size,
        update_coefficient=update_coefficient,
        corr_coeff=corr_coeff,
    )
)

print("final string:  ", tuple(final_string_t.shape), tuple(final_string_r.shape))
print("final walkers: ", tuple(walker_t.shape), tuple(walker_r.shape))

## 9. Save NumPy and molecular trajectory outputs

In [ ]:
initial_t_cpu = initial_t.detach().cpu()
initial_r_cpu = initial_r.detach().cpu()
final_string_t_cpu = final_string_t.detach().cpu()
final_string_r_cpu = final_string_r.detach().cpu()
walker_t_cpu = walker_t.detach().cpu()
walker_r_cpu = walker_r.detach().cpu()

npz_path = output_dir / "dynamic_se3_chignolin_samples.npz"
np.savez_compressed(
    npz_path,
    sequence=sequence,
    model_name=model_name,
    initial_pos=initial_t_cpu.numpy(),
    initial_node_orientations=initial_r_cpu.numpy(),
    final_string_pos=final_string_t_cpu.numpy(),
    final_string_node_orientations=final_string_r_cpu.numpy(),
    walker_pos=walker_t_cpu.numpy(),
    walker_node_orientations=walker_r_cpu.numpy(),
)

flat_walker_t = walker_t_cpu.reshape(-1, len(sequence), 3)
flat_walker_r = walker_r_cpu.reshape(-1, len(sequence), 3, 3)

save_pdb_and_xtc(
    pos_nm=final_string_t_cpu,
    node_orientations=final_string_r_cpu,
    sequence=sequence,
    topology_path=output_dir / "string_topology.pdb",
    xtc_path=output_dir / "string.xtc",
    filter_samples=False,
)
save_pdb_and_xtc(
    pos_nm=flat_walker_t,
    node_orientations=flat_walker_r,
    sequence=sequence,
    topology_path=output_dir / "walkers_topology.pdb",
    xtc_path=output_dir / "walkers.xtc",
    filter_samples=False,
)

print(f"saved arrays:  {npz_path}")
print(f"saved string:  {output_dir / 'string.xtc'}")
print(f"saved walkers: {output_dir / 'walkers.xtc'}")

## 10. Visualize the C-alpha string

Each line is one point along the final string, drawn through that structure's C-alpha positions.

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

for index, frame in enumerate(final_string_t_cpu.numpy()):
    ax.plot(
        frame[:, 0],
        frame[:, 1],
        frame[:, 2],
        marker="o",
        alpha=0.75,
        label=f"point {index}",
    )

ax.set_xlabel("x [nm]")
ax.set_ylabel("y [nm]")
ax.set_zlabel("z [nm]")
ax.set_title("BioEmu dynamic SE(3) string for Chignolin")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout()
plt.show()